In [1]:
import os
import pickle
from pathlib import Path

import pillow_heif
from PIL import Image
from torch.utils.data import DataLoader

import torch
from torch import optim

import tiktoken
from datasets import Dataset, concatenate_datasets
from tokenizers import Tokenizer
from transformers import (
    VisionEncoderDecoderModel,
    TrOCRProcessor,
    AutoImageProcessor,
    PreTrainedTokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

/home/luchian/prog/self_prog/Projects/TextRecognition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pillow_heif.register_heif_opener()

In [3]:
#main config
DATA_ROOT = "/home/luchian/all_data/datasets/Image_data.v4" 
OUTPUT_DIR = "/home/luchian/all_data/ML_models_weights/ViT_training"

# Training hyperparameters
MODEL_NAME = "microsoft/trocr-base-handwritten"
BATCH_SIZE = 5
EPOCHS = 3
LR = 1e-9
MAX_TARGET_LENGTH = 76

In [4]:
with open('/home/luchian/all_data/ML_models_artifacts/tokenizer.pkl','rb') as file:
    main_tokenizer = pickle.load(file)

In [5]:
main_tokenizer.eos_token_id = main_tokenizer.get_vocab()['<eos>']
main_tokenizer.bos_token_id = main_tokenizer.get_vocab()['<sos>']

In [6]:
def make_spaces(word):
    """Makes spaces between the letter of a given word"""
    return ' '.join(list(word))

In [7]:
image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

processor = TrOCRProcessor(
    image_processor=image_processor,
    tokenizer=main_tokenizer
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# update model config
model.decoder.resize_token_embeddings(len(main_tokenizer.get_vocab()))
model.config.decoder.vocab_size = len(main_tokenizer.get_vocab())
model.config.decoder_start_token_id = main_tokenizer.get_vocab()['<sos>']
model.config.eos_token_id = main_tokenizer.get_vocab()['<eos>']
model.config.pad_token_id = main_tokenizer.get_vocab()['<pad>']

In [9]:
def load_dataset_from_folders(split_prefix):
    """
    split_prefix: 0=train, 1=val, 2=test
    """
    items = []
    for folder in Path(DATA_ROOT).glob(f"{split_prefix}.*"):
        target_name = folder.name.split(".", 1)[1]  # everything after first dot
        for img_path in folder.glob("*.*"):  # all images
            items.append({
                "pixel_values": str(img_path),
                "labels": make_spaces(target_name)
            })
    return Dataset.from_list(items)

In [10]:
def collate_fn(batch):
    #if it is a list we convert it to list of dicts
    images = [Image.open(b["pixel_values"]).convert("RGB") for b in batch]
    pixel_values = processor.image_processor(images=images, return_tensors="pt").pixel_values
    labels = main_tokenizer(
        [b["labels"] for b in batch],
        padding="max_length",
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt"
    ).input_ids
    #enable gradient for tensors
    # labels[labels==0] = -100
    labels.to(dtype=torch.long)
    pixel_values.requires_grad = True
    return {"pixel_values": pixel_values, "labels": labels}

In [11]:
train_ds = load_dataset_from_folders(0)
val_ds   = load_dataset_from_folders(1)
test_ds  = load_dataset_from_folders(2)

print("Training samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Test samples:", len(test_ds), end='\n\n')


#splitting validation in two parts
half_val = int(len(val_ds)/2.0)
train_ds = concatenate_datasets([train_ds, Dataset.from_dict(val_ds[:half_val])])
val_ds = Dataset.from_dict(val_ds[half_val:])
#printing stats again
print("Training samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Test samples:", len(test_ds), end='\n\n')

Training samples: 6643
Validation samples: 1020
Test samples: 2914

Training samples: 7153
Validation samples: 510
Test samples: 2914



In [12]:
def levenshtein(a, b):
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    prev = list(range(m+1))
    for i, ca in enumerate(a, start=1):
        curr = [i]+[0]*m
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca==cb else 1
            curr[j] = min(prev[j]+1, curr[j-1]+1, prev[j-1]+cost)
        prev = curr
    return prev[m]

In [13]:
def cer_metric(eval_pred):
    """
    parameters
    -----------
    eval_pred: tuple
        tuple contaiing 2-d tensors with indecies of vocabulary
        first tensor contains true indecies 
        second tensor contains prdiction indecies
    """
    preds, labels = eval_pred.predictions, eval_pred.label_ids
    decoded_preds = [''.join(str.split(' ')) for str in main_tokenizer.batch_decode(preds, skip_special_tokens=True)]
    labels = [[l if l!=-100 else main_tokenizer.pad_token_id for l in ls] for ls in labels]
    decoded_labels = [''.join(str.split(' ')) for str in main_tokenizer.batch_decode(labels, skip_special_tokens=True)]
    total_err = 0
    total_chars = 0
    for p,r in zip(decoded_preds, decoded_labels):
        total_err += levenshtein(p,r)
        total_chars += max(1,len(r))
    return {"cer": float(total_err/total_chars)}

In [14]:
from peft import get_peft_model, LoraConfig, TaskType

config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 761,856 || all params: 283,253,248 || trainable%: 0.2690


In [15]:
#freeze encoder 

for param in model.encoder.parameters():
    param.requires_grad = False

In [16]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="epoch",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    fp16=True,
    logging_strategy='steps',
    logging_steps=100,
    learning_rate=1e-9,
    report_to='none',
    save_total_limit=3,
    do_eval=False,
    do_train=True
)

In [17]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    processing_class=processor,
    compute_metrics=cer_metric)

In [18]:
model.print_trainable_parameters()

trainable params: 761,856 || all params: 283,253,248 || trainable%: 0.2690


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2, 'pad_token_id': 0}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,15.065300


In [ ]:
def inference(image, device='cpu', model=model):
    model.eval()
    model.to(device=device)
    img = Image.open(image)
    pixel_values = processor(images=img, return_tensors="pt").pixel_values.to(device)

    generated_ids = model.generate(
        pixel_values,
        max_length=100,
        bos_token_id=processor.tokenizer.get_vocab()['<sos>'],
        eos_token_id=processor.tokenizer.get_vocab()['<eos>'],
        pad_token_id=processor.tokenizer.get_vocab()['<pad>'],
        do_sample=True
    )

    text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text

40